<a href="https://colab.research.google.com/github/pritam01729/Customer-Churn-Prediction/blob/main/movie_reco.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

'''
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
'''


"\nimport os\nfor dirname, _, filenames in os.walk('/kaggle/input'):\n    for filename in filenames:\n        print(os.path.join(dirname, filename))\n"

In [2]:
movies = pd.read_csv('/content/tmdb_5000_movies.csv')
credits = pd.read_csv('/content/tmdb_5000_credits.csv')

In [3]:
#movies was causing some problem as it was getting selected as series (not a dataframe). So check it first
print(type(movies))

<class 'pandas.core.frame.DataFrame'>


**Basics checking ----------**

In [4]:
movies.head(2)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500


In [5]:
movies.shape

(4803, 20)

In [6]:
credits.head(2)

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


In [7]:
credits.shape

(4803, 4)

In [8]:

movies = movies.merge(credits,on='title')

In [9]:
movies.shape

(4809, 23)

In [10]:
movies.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,movie_id,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,19995,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [11]:
#checking whether movies is a datframe or a series
print(type(movies))

<class 'pandas.core.frame.DataFrame'>


Now we will think about tags. cosider the attributes -- movie_id, title, overview, generes, keywords, cast, crew

In [12]:
#most of the language is eng. So 'original_language' not that much importent for tag creation
movies['original_language'].value_counts()


,count
original_language,
en,4510
fr,70
es,32
zh,27
de,27
hi,19
ja,16
it,14
ko,12


In [13]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4809 entries, 0 to 4808
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4809 non-null   int64  
 1   genres                4809 non-null   object 
 2   homepage              1713 non-null   object 
 3   id                    4809 non-null   int64  
 4   keywords              4809 non-null   object 
 5   original_language     4809 non-null   object 
 6   original_title        4809 non-null   object 
 7   overview              4806 non-null   object 
 8   popularity            4809 non-null   float64
 9   production_companies  4809 non-null   object 
 10  production_countries  4809 non-null   object 
 11  release_date          4808 non-null   object 
 12  revenue               4809 non-null   int64  
 13  runtime               4807 non-null   float64
 14  spoken_languages      4809 non-null   object 
 15  status               

In [14]:
#To check Dtaframe or series we can use type(object)  OR-----
# Check if it is a DataFrame
if isinstance(movies, pd.DataFrame):
    print("It is a DataFrame")

# Check if it is a Series
elif isinstance(movies, pd.Series):
    print("It is a Series")

It is a DataFrame


just select those required attributes to work on

In [15]:
movies = movies[['id','title','overview','genres','keywords','cast','crew']]

In [16]:
movies.head(1)

,id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


Now we want to create a new datframe - id, title, tags
tags will consist of the col - overview, genres, keywords, cast (choose top cast members), crew (choose only director)

In [17]:
movies.isnull().sum()

,0
id,0
title,0
overview,3
genres,0
keywords,0
cast,0
crew,0


In [18]:
#In the overview col, there are 3 null val. So we will just drop those rows
movies.dropna(inplace=True)

In [19]:
movies.isnull().sum()

,0
id,0
title,0
overview,0
genres,0
keywords,0
cast,0
crew,0


Now before creating tags, I want to put 'genres' (dictionary format with string keys) in a suitable list format.

For that we create a convert() fun. But keys has to be int (not string). So we import **ast** module ---
**ast.literal_eval(node_or_string)**: Safely evaluates strings containing Python literals (like strings, numbers, tuples, lists, dicts) without running arbitrary code, eliminating the security risks of eval()

In [20]:
movies['genres'][0]

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

See one movie can have many genres -- like Adventure, Action.... and each of the genre has a particular ID. So movies['genres'][i] is a dictionary
We want to convert them into list format removing the IDs

In [21]:
import ast

def convert(obj):
  L= []
  for i in ast.literal_eval(obj):
    L.append(i['name'])
  return L

In [22]:
#.apply() is the standard and most idiomatic way in pandas to run a custom function on every individual element in a Series
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)
movies.head()

,id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


Again I want to put 'cast' (dictionary format with string keys) in a suitable list format with maximum 3 elts.

In [23]:
def convert3(text):
    L = []
    counter = 0
    for i in ast.literal_eval(text):
        if counter < 3:
            L.append(i['name'])
        counter+=1
    return L

In [24]:
movies['cast'] = movies['cast'].apply(convert3)

I want crew only to be director

In [25]:
def fetch_director(text):
    L = []
    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            L.append(i['name'])
    return L

In [26]:
movies['crew'] = movies['crew'].apply(fetch_director)

In [27]:
movies.head(1)

,id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]


To create tags we needed 'genres', 'keywords', 'cast', 'crew' and 'overview' to be concatinated in a list format.
So we split the sentenceses of 'overview' col into words and store in a list

In [28]:
movies['overview'] = movies['overview'].apply(lambda x:x.split())

#lambda x: specifies the input argument. For every row, x will represent the text string stored in that row of the 'overview' column.
#x.split(): is Python’s built-in string method. When called with no arguments, it splits a string wherever it finds whitespace (spaces, tabs, newlines) and returns a list of individual words.

In [29]:
movies['overview'].head(5)

,overview
0,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,"[Captain, Barbossa,, long, believed, to, be, d..."
2,"[A, cryptic, message, from, Bond’s, past, send..."
3,"[Following, the, death, of, District, Attorney..."
4,"[John, Carter, is, a, war-weary,, former, mili..."


Now we will use a transformation on the lists of those attributes to merge all the words separated by space. Coz -- Sam Walder and Sam Jennifer may create confusion

In [30]:
def collapse(L):
    L1 = []
    for i in L:
        L1.append(i.replace(" ",""))
    return L1

In [31]:
movies['cast'] = movies['cast'].apply(collapse)
movies['crew'] = movies['crew'].apply(collapse)
movies['genres'] = movies['genres'].apply(collapse)
movies['keywords'] = movies['keywords'].apply(collapse)

In [32]:
movies.head(2)

,id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron]
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drugabuse, exoticisland, eastindiatrad...","[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski]


In [33]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [34]:
#ctreating a new datfarme
new_df = movies[['id', 'title', 'tags']]

In [35]:
new_df['tags'][0]

['In',
 'the',
 '22nd',
 'century,',
 'a',
 'paraplegic',
 'Marine',
 'is',
 'dispatched',
 'to',
 'the',
 'moon',
 'Pandora',
 'on',
 'a',
 'unique',
 'mission,',
 'but',
 'becomes',
 'torn',
 'between',
 'following',
 'orders',
 'and',
 'protecting',
 'an',
 'alien',
 'civilization.',
 'Action',
 'Adventure',
 'Fantasy',
 'ScienceFiction',
 'cultureclash',
 'future',
 'spacewar',
 'spacecolony',
 'society',
 'spacetravel',
 'futuristic',
 'romance',
 'space',
 'alien',
 'tribe',
 'alienplanet',
 'cgi',
 'marine',
 'soldier',
 'battle',
 'loveaffair',
 'antiwar',
 'powerrelations',
 'mindandsoul',
 '3d',
 'SamWorthington',
 'ZoeSaldana',
 'SigourneyWeaver',
 'JamesCameron']

In [36]:
#converting the tags -- list to string
new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))

/tmp/ipykernel_604/3903689613.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))


In [37]:
new_df['tags'][0]

'In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. Action Adventure Fantasy ScienceFiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d SamWorthington ZoeSaldana SigourneyWeaver JamesCameron'

In [38]:
new_df.head()

,id,title,tags
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...
4,49529,John Carter,"John Carter is a war-weary, former military ca..."


We will use two technique for recommendations ------------------

First ---

**Word2Vec or GloVe:** These shallow neural networks learn word associations, allowing your system to understand that movies with slightly different tags might still be highly similar.

In [40]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 25.3 MB/s eta 0:00:00


In [41]:
import gensim.downloader as api

In [42]:
# Download and load a pre-trained GloVe model (100-dimensional word vectors)
print("Loading pre-trained GloVe model...")
glove = api.load('glove-wiki-gigaword-100')

Loading pre-trained GloVe model...
[==================================================] 100.0% 128.1/128.1MB downloaded


In [43]:
# Function to compute the average semantic vector for a movie's tags
def get_movie_vector(text):
    # Find all words in the tags that exist in the GloVe vocabulary
    words = [word for word in text.lower().split() if word in glove]
    if len(words) == 0:
        return np.zeros(100)
    # Average the vectors of all matched words
    return np.mean(glove[words], axis=0)

In [44]:
# Generate the 2D vector matrix for all movies
vector_glove = np.array(new_df['tags'].apply(get_movie_vector).tolist())

print("Vector matrix shape:", vector_glove.shape)

Vector matrix shape: (4806, 100)


In [45]:
# Calculate Cosine Similarity on your new neural embeddings
from sklearn.metrics.pairwise import cosine_similarity
similarity_glove = cosine_similarity(vector_glove)

We now create a function to recommend 5 top similar movies

In [46]:
def recommend(movie):
  movie_index = new_df[new_df['title']== movie].index[0]
  similarity_vec1 = similarity_glove[movie_index]
  movie_list = sorted(list(enumerate(similarity_vec1)), reverse = True, key=lambda x:x[1])[1:6]

  for i in movie_list:    #i is a list which conatins [index, score]
    #print(i[0])
    print(new_df.iloc[i[0]].title)

In [47]:
# Run your existing recommend function
recommend('Avatar')

Jupiter Ascending
Titan A.E.
Aliens vs Predator: Requiem
Starship Troopers
Valiant


**Vectorization** of each movies based on tags -- [we collect 5000 unique words from all the tags (after stemming) and create vectors of dim 5000 for each movies]

We see there are a lot of similar words like -- [ loved, loving, love ] So we will apply **stemming** on each tags

In [48]:
#NLTK, or the Natural Language Toolkit, is a popular open-source Python library used for Natural Language Processing (NLP) tasks,
#allowing computers to analyze, preprocess, and understand human language data
import nltk
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [49]:
def stem(text):
  y = []
  for i in text.split():  #.split() method takes a single string and breaks it apart into a list of smaller strings.
    y.append(ps.stem(i))
  return " ".join(y)
#.join() takes a list of strings and stitches them back together into a single string
#" " (The Glue): The string that comes before the .join() is the separator. In your case, it is a single space. This tells Python, "Put a space between every item when you stick them together.

In [50]:
#Example--
stem("In the 22nd century, a paraplegic loving")

'in the 22nd century, a parapleg love'

In [51]:
new_df['tags'] = new_df['tags'].apply(stem)

/tmp/ipykernel_604/3213734980.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(stem)


In [52]:
new_df['tags'][0]

'in the 22nd century, a parapleg marin is dispatch to the moon pandora on a uniqu mission, but becom torn between follow order and protect an alien civilization. action adventur fantasi sciencefict cultureclash futur spacewar spacecoloni societi spacetravel futurist romanc space alien tribe alienplanet cgi marin soldier battl loveaffair antiwar powerrel mindandsoul 3d samworthington zoesaldana sigourneyweav jamescameron'

Now go for vectorization ----

In [53]:
#Convert a collection of text documents to a matrix of token counts
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000,stop_words='english')

In [54]:
#fit_transform(raw_documents[, y]) - Learn the vocabulary dictionary and return term-document matrix.
vector_cv = cv.fit_transform(new_df['tags']).toarray()

In [55]:
vector_cv.shape

(4806, 5000)

In [56]:
print(vector_cv)

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


In [57]:
cv.get_feature_names_out()

array(['000', '007', '10', ..., 'zone', 'zoo', 'zooeydeschanel'],
      dtype=object)

Now we will define cosine similarity of movies. We will create a similarity matrix for all 4800 movies

In [58]:
similarity_cv = cosine_similarity(vector_cv)

We need similarity[0] (all the cosine similarity score of all movies with the 0th-indexed movie) in descending order

In [59]:
sorted(similarity_cv[0], reverse = True)

[np.float64(1.0000000000000002),
 np.float64(0.28676966733820225),
 np.float64(0.26901379342448517),
 np.float64(0.2605130246476754),
 np.float64(0.255608593705383),
 np.float64(0.25038669783359574),
 np.float64(0.24511108480187255),
 np.float64(0.24455799402225922),
 np.float64(0.23179316248638276),
 np.float64(0.23174488732966075),
 np.float64(0.2278389747471728),
 np.float64(0.2252817784447915),
 np.float64(0.22269966704152225),
 np.float64(0.21853668936906193),
 np.float64(0.21239769762143662),
 np.float64(0.2108663315950723),
 np.float64(0.2105263157894737),
 np.float64(0.20443988269091456),
 np.float64(0.20437977982832192),
 np.float64(0.20395079136182276),
 np.float64(0.2029530274475215),
 np.float64(0.2029530274475215),
 np.float64(0.20277677641345318),
 np.float64(0.2024645717996314),
 np.float64(0.2020475485519274),
 np.float64(0.1979082783981174),
 np.float64(0.19767387315371682),
 np.float64(0.1976738731537168),
 np.float64(0.19672236884115843),
 np.float64(0.19252140716412

But see here we are loosing the indexed positions while sorting... so we use enumerate

In [60]:
#list(enumerate(similarity_cv[0]))
#sorted(list(enumerate(similarity_cv[0])), reverse = True)  -- this sorts on the basis of enumerated index

#we want to sort on th basis of similarity values  and only consider the first 5 similar movies (except the movie itself) -- so [1:6]
sorted(list(enumerate(similarity_cv[0])), reverse = True, key=lambda x:x[1])[1:6]

[(1214, np.float64(0.28676966733820225)),
 (2405, np.float64(0.26901379342448517)),
 (3728, np.float64(0.2605130246476754)),
 (507, np.float64(0.255608593705383)),
 (539, np.float64(0.25038669783359574))]

In [61]:
#how to get a index of a movie
print(new_df[new_df['title']=='Batman Begins'].index[0])

119


We now create a function to recommend 5 top similar movies

In [62]:
def recommend(movie):
  movie_index = new_df[new_df['title']== movie].index[0]
  similarity_vec2 = similarity_cv[movie_index]
  movie_list = sorted(list(enumerate(similarity_vec2)), reverse = True, key=lambda x:x[1])[1:6]

  for i in movie_list:    #i is a list which conatins [index, score]
    #print(i[0])
    print(new_df.iloc[i[0]].title)

In [63]:
recommend('Avatar')

Aliens vs Predator: Requiem
Aliens
Falcon Rising
Independence Day
Titan A.E.
